In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import (col,sum as _sum,count,year,month,dayofmonth)

In [3]:
spark = SparkSession.builder.appName("LocalDataWarehouseDemo").config("spark.sql.warehouse.dir","./warehouse").enableHiveSupport().getOrCreate()
print("Spark Session Started")

Spark Session Started


In [5]:
spark.sql("create database if not exists ecommerce_dw")
spark.sql("use ecommerce_dw")

DataFrame[]

In [10]:
customers_df = spark.read.option('header', True).option('inferSchema',True).csv('./data/customers.csv')
print('customers')
customers_df.show()
products_df = spark.read.option('header', True).option('inferSchema',True).csv('./data/products.csv')
print('products')
products_df.show()
sales_df = spark.read.option('header', True).option('inferSchema',True).csv('./data/sales.csv')
print('sales')
sales_df.show()

customers
+-----------+-------------+---------+----------+
|customer_id|customer_name|     city|     state|
+-----------+-------------+---------+----------+
|       C001|        Rahul|Bangalore| Karnataka|
|       C002|        Priya|  Chennai|Tamil Nadu|
|       C003|         Arun|Hyderabad| Telangana|
|       C004|        Meena|Bangalore| Karnataka|
|       C005|        Theeb|Banglaore|Karanataka|
+-----------+-------------+---------+----------+

products
+----------+------------+-----------+-----+
|product_id|product_name|   category|price|
+----------+------------+-----------+-----+
|      P001|      Laptop|Electronics|60000|
|      P002|       Mouse|Electronics| 1000|
|      P003|    Keyboard|Electronics| 2000|
|      P004|       Chair|  Furniture| 5000|
|      P005|       Table|  Furniture|10000|
+----------+------------+-----------+-----+

sales
+--------+-----------+----------+--------+----------+
|order_id|customer_id|product_id|quantity|order_date|
+--------+-----------+------

In [16]:
sales_df = sales_df.filter(col('order_id').isNotNull()).filter(col('customer_id').isNotNull()).filter(col('product_id').isNotNull()).filter(col('quantity') > 0)

In [17]:
sales_df.show()

+--------+-----------+----------+--------+----------+
|order_id|customer_id|product_id|quantity|order_date|
+--------+-----------+----------+--------+----------+
|    O001|       C001|      P001|       1|2026-10-01|
|    O002|       C002|      P002|       2|2026-10-01|
|    O003|       C003|      P003|       1|2026-10-01|
|    O004|       C001|      P004|       2|2026-10-02|
|    O005|       C004|      P005|       1|2026-10-02|
|    O006|       C002|      P001|       1|2026-10-02|
|    O007|       C003|      P002|       3|2026-10-03|
|    O008|       C004|      P003|       2|2026-10-03|
|    O009|       C001|      P002|       1|2026-10-03|
|    O010|       C002|      P004|       1|2026-10-04|
+--------+-----------+----------+--------+----------+



In [21]:
dim_customer = customers_df.select("customer_id","customer_name","city","state")
dim_customer.show()
dim_product = products_df.select("product_id","product_name","category","price")
dim_product.show()

+-----------+-------------+---------+----------+
|customer_id|customer_name|     city|     state|
+-----------+-------------+---------+----------+
|       C001|        Rahul|Bangalore| Karnataka|
|       C002|        Priya|  Chennai|Tamil Nadu|
|       C003|         Arun|Hyderabad| Telangana|
|       C004|        Meena|Bangalore| Karnataka|
|       C005|        Theeb|Banglaore|Karanataka|
+-----------+-------------+---------+----------+

+----------+------------+-----------+-----+
|product_id|product_name|   category|price|
+----------+------------+-----------+-----+
|      P001|      Laptop|Electronics|60000|
|      P002|       Mouse|Electronics| 1000|
|      P003|    Keyboard|Electronics| 2000|
|      P004|       Chair|  Furniture| 5000|
|      P005|       Table|  Furniture|10000|
+----------+------------+-----------+-----+



In [26]:
fact_sales = sales_df.join(dim_product,'product_id','inner') 
fact_sales = fact_sales.withColumn('total_amount',col('quantity')* col('price'))
fact_sales.show()

+----------+--------+-----------+--------+----------+------------+-----------+-----+------------+
|product_id|order_id|customer_id|quantity|order_date|product_name|   category|price|total_amount|
+----------+--------+-----------+--------+----------+------------+-----------+-----+------------+
|      P001|    O001|       C001|       1|2026-10-01|      Laptop|Electronics|60000|       60000|
|      P002|    O002|       C002|       2|2026-10-01|       Mouse|Electronics| 1000|        2000|
|      P003|    O003|       C003|       1|2026-10-01|    Keyboard|Electronics| 2000|        2000|
|      P004|    O004|       C001|       2|2026-10-02|       Chair|  Furniture| 5000|       10000|
|      P005|    O005|       C004|       1|2026-10-02|       Table|  Furniture|10000|       10000|
|      P001|    O006|       C002|       1|2026-10-02|      Laptop|Electronics|60000|       60000|
|      P002|    O007|       C003|       3|2026-10-03|       Mouse|Electronics| 1000|        3000|
|      P003|    O008

In [27]:
fact_sales = fact_sales.select('order_id','customer_id','product_id','order_date','quantity','price','total_amount')

In [31]:
dim_date = sales_df.select('order_date').distinct()
dim_date = dim_date.withColumn("year", year("order_date")).withColumn("month", month("order_date")).withColumn("day", dayofmonth("order_date"))
dim_date.show()

+----------+----+-----+---+
|order_date|year|month|day|
+----------+----+-----+---+
|2026-10-01|2026|   10|  1|
|2026-10-03|2026|   10|  3|
|2026-10-04|2026|   10|  4|
|2026-10-02|2026|   10|  2|
+----------+----+-----+---+



In [34]:
dim_customer.write.mode('overwrite').saveAsTable('dim_customer')
dim_product.write.mode('overwrite').saveAsTable('dim_product')
dim_date.write.mode('overwrite').saveAsTable('dim_date')

In [35]:
fact_sales.write.mode('overwrite').saveAsTable('fact_sales')

In [38]:
daily_sales_summary = fact_sales.join(dim_product,'product_id')
daily_sales_summary = daily_sales_summary.groupBy('order_date','category')
daily_sales_summary = daily_sales_summary.agg(
    _sum("total_amount").alias("daily_revenue"),
    _sum("quantity").alias("total_units_sold"),
    count("order_id").alias("number_of_orders")
)
daily_sales_summary.write.mode('overwrite').saveAsTable('daily_sales_summary')

In [39]:
daily_sales_summary.show()

+----------+-----------+-------------+----------------+----------------+
|order_date|   category|daily_revenue|total_units_sold|number_of_orders|
+----------+-----------+-------------+----------------+----------------+
|2026-10-02|  Furniture|        20000|               3|               2|
|2026-10-01|Electronics|        64000|               4|               3|
|2026-10-04|  Furniture|         5000|               1|               1|
|2026-10-03|Electronics|         8000|               6|               3|
|2026-10-02|Electronics|        60000|               1|               1|
+----------+-----------+-------------+----------------+----------------+



In [41]:
print('tables in data warehouse')
spark.sql('show tables').show()

tables in data warehouse
+------------+-------------------+-----------+
|   namespace|          tableName|isTemporary|
+------------+-------------------+-----------+
|ecommerce_dw|daily_sales_summary|      false|
|ecommerce_dw|       dim_customer|      false|
|ecommerce_dw|           dim_date|      false|
|ecommerce_dw|        dim_product|      false|
|ecommerce_dw|         fact_sales|      false|
+------------+-------------------+-----------+



In [43]:
spark.sql('select sum(daily_revenue) as total_revenue from daily_sales_summary').show()

+-------------+
|total_revenue|
+-------------+
|       157000|
+-------------+



In [44]:
spark.sql('''select category,sum(daily_revenue) as revenue 
from daily_sales_summary
group by category
order by revenue desc
''').show()

+-----------+-------+
|   category|revenue|
+-----------+-------+
|Electronics| 132000|
|  Furniture|  25000|
+-----------+-------+



In [45]:
spark.sql('''select customer_id,sum(total_amount) as c_revenue 
from fact_sales
group by customer_id
order by c_revenue desc
''').show()

+-----------+---------+
|customer_id|c_revenue|
+-----------+---------+
|       C001|    71000|
|       C002|    67000|
|       C004|    14000|
|       C003|     5000|
+-----------+---------+



In [46]:
spark.sql('''select order_date,sum(total_amount) as o_revenue 
from fact_sales
group by order_date
order by o_revenue desc
''').show()

+----------+---------+
|order_date|o_revenue|
+----------+---------+
|2026-10-02|    80000|
|2026-10-01|    64000|
|2026-10-03|     8000|
|2026-10-04|     5000|
+----------+---------+



In [48]:
spark.sql('''select c.city,sum(f.total_amount) as citywise_revenue 
from fact_sales f
join dim_customer c
on f.customer_id = c.customer_id
group by c.city
order by citywise_revenue desc
''').show()

+---------+----------------+
|     city|citywise_revenue|
+---------+----------------+
|Bangalore|           85000|
|  Chennai|           67000|
|Hyderabad|            5000|
+---------+----------------+



In [49]:
spark.stop()